In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import col, explode_outer, to_timestamp


@dp.table(
    name="gharchive_silver",
    comment="Flattened GitHub Archive events with exploded commits"
)
def gharchive_silver():
    # dp.read() automatically registers dependency on gharchive_bronze
    df = dp.read("gharchive_bronze")

    # Flatten repo struct
    if "repo" in df.columns:
        df = (df
            .withColumn("repo_id", col("repo.id"))
            .withColumn("repo_name", col("repo.name"))
            .withColumn("repo_url", col("repo.url"))
        )

    # Flatten actor struct
    if "actor" in df.columns:
        df = (df
            .withColumn("actor_id", col("actor.id"))
            .withColumn("actor_login", col("actor.login"))
            .withColumn("actor_display_login", col("actor.display_login"))
            .withColumn("actor_avatar_url", col("actor.avatar_url"))
        )

    # Explode payload.commits into separate rows
    df = df.withColumn("commit", explode_outer("payload.commits"))

    # Flatten commit struct
    df = (df
        .withColumn("commit_sha", col("commit.sha"))
        .withColumn("commit_message", col("commit.message"))
        .withColumn("commit_author_name", col("commit.author.name"))
        .withColumn("commit_author_email", col("commit.author.email"))
    )

    # Parse created_at to timestamp
    df = df.withColumn("created_at", to_timestamp("created_at"))

    return df.drop("repo", "actor", "payload", "commit", "org")